In [1]:
import numpy as np
import pandas as pd
from scprint import scPrint
from huggingface_hub import hf_hub_download
import bionty as bt
import lamindb as ln
import os

# local .py file
from scPRINT import (
    load_scprint_model,
    create_gene_mapping_table,
    extract_attention_weights,
    extract_gene_embeddings,
    extract_model_metadata,
    format_weights_for_saving,
    populate_lamin_db
)

# Configuration
DATA_DIR = "data"
MODEL_PATH = os.path.join(DATA_DIR, "scPRINT")
os.makedirs(MODEL_PATH, exist_ok=True)

# ============================================================
# Main Extraction Workflow
# ============================================================

print("\n" + "="*60)
print("Extracting: scPRINT")
print("="*60)

# 0. Populate the lamin database
populate_lamin_db()

# 1. Download and load model
print("\n1. Downloading/loading model...")
model_checkpoint_file = hf_hub_download(
    repo_id="jkobject/scPRINT", 
    filename="v2-medium.ckpt", 
    cache_dir=MODEL_PATH
)

model = load_scprint_model(model_checkpoint_file, transformer="normal")

# 2. Load gene table with Ensembl IDs
print("2. Loading gene mapping from lamindb...")
gene_table = create_gene_mapping_table(model)
n_genes = len(gene_table)
n_layers = model.nlayers
print(f"   {n_genes} genes, {n_layers} layers")

# 3. Extract components
print("3. Extracting weights...")
embeddings = extract_gene_embeddings(model, n_genes)
print(f"   Embeddings: {embeddings.shape}")

attention_weights = extract_attention_weights(model, n_layers)
print(f"   Attention weights: {n_layers} layers × 4 matrices (Q,K,V,O)")

model_metadata = extract_model_metadata(model, "scPRINT", n_genes)

# 4. Format for saving
print("4. Formatting data...")
weights_dict = format_weights_for_saving(embeddings, attention_weights)


No module named 'triton'
FlashAttention is not installed, not using it..


/Users/sean/Desktop/GITHUB/napistu/lib/napistu-scrapyard/applications/foundation_models/.scprint/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


→ connected lamindb: anonymous/scPRINT_lamin


Mon Oct  6 11:55:44 2025 INFO Lamin database already configured



Extracting: scPRINT

1. Downloading/loading model...
RuntimeError caught: scPrint is not attached to a `Trainer`.
2. Loading gene mapping from lamindb...
   44756 genes, 8 layers
3. Extracting weights...
   Embeddings: (44756, 256)
   Attention weights: 8 layers × 4 matrices (Q,K,V,O)
4. Formatting data...
